In [1]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F 
import torch.distributions as distributions 
import torch.optim as optim 

import numpy as np
import matplotlib.pyplot as plt 
import gymnasium as gym

In [2]:
train_env = gym.make("CartPole-v1")
val_env = gym.make("CartPole-v1")
test_env = gym.make("CartPole-v1")

In [3]:
class ReplayBuffer():
    def __init__(self, capacity, n_dim, device = "cpu"):
        self.capacity = capacity 
        self.n_dim = n_dim 
        self.device = device 
        self.state_buff = np.zeros((capacity, *n_dim), dtype = np.float32)
        self.action_buff = np.zeros((capacity, ), dtype= np.int64)
        self.nstate_buff = np.zeros((capacity, *n_dim), dtype = np.float32)
        self.reward_buff = np.zeros((capacity, ), dtype = np.float32)
        self.done_buff = np.zeros((capacity,), dtype = np.float32)
        self.truncated_buff = np.zeros((capacity,), dtype = np.float32)
        self.size = 0
        self.ptr = 0 

    def store(self, state, action, next_state, reward, done, truncated):
        self.state_buff[self.ptr] = state 
        self.action_buff[self.ptr] = action 
        self.nstate_buff[self.ptr] = next_state 
        self.reward_buff[self.ptr] = reward 
        self.done_buff[self.ptr] = done 
        self.truncated_buff[self.ptr] = truncated 
        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample_batch(self, batch_size):
        idxs = np.random.choice(self.size, batch_size)
        batch = [
            self.state_buff[idxs],
            self.action_buff[idxs],
            self.nstate_buff[idxs],
            self.reward_buff[idxs],
            self.done_buff[idxs],
            self.truncated_buff[idxs],
        ]
        return [torch.tensor(x, device = self.device) for x in batch]


In [4]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        return self.net(x)

In [5]:
class ActorCritic(nn.Module):
    def __init__(self, actor, critic):
        super().__init__()
        self.actor = actor 
        self.critic = critic 

    def forward(self, state):
        action_pred = self.actor(state)
        value_pred = self.critic(state)
        return action_pred, value_pred 

In [6]:
INPUT_DIM = train_env.observation_space.shape[0]
HIDDEN_DIM = 128 
OUTPUT_DIM = train_env.action_space.n 

actor = MLP(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM)
critic = MLP(INPUT_DIM, HIDDEN_DIM, 1)
policy = ActorCritic(actor, critic)
policy

ActorCritic(
  (actor): MLP(
    (net): Sequential(
      (0): Linear(in_features=4, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=2, bias=True)
    )
  )
  (critic): MLP(
    (net): Sequential(
      (0): Linear(in_features=4, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=1, bias=True)
    )
  )
)

In [7]:
LEARNING_RATE = 1e-3
optimizer = optim.Adam(policy.parameters(), lr = LEARNING_RATE)
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)

In [8]:
def init_weight(m):
    if type(m) == nn.Linear:
        torch.nn.init.orthogonal_(m.weight)
        m.bias.data.fill_(0)

policy.apply(init_weight)


ActorCritic(
  (actor): MLP(
    (net): Sequential(
      (0): Linear(in_features=4, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=2, bias=True)
    )
  )
  (critic): MLP(
    (net): Sequential(
      (0): Linear(in_features=4, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=1, bias=True)
    )
  )
)

In [9]:
def soft_update(net, target_net, tau):
    for param, target_param in zip(net.parameters(), target_net.parameters()):
        target_param.data.copy_(tau * param.data + (1 - tau) * target_param.data)

In [10]:
def train(env, policy, target_policy, optimizer, buffer, batch_size, discounted_factor, beta, weight_clip, tau):
    episode_rewards = 0 
    episode_actor_loss = 0
    episode_critic_loss = 0
    done, truncated = False, False 

    state, _ = env.reset()
    
    while not done and not truncated: 
        state = torch.tensor(state).unsqueeze(0)
        with torch.no_grad():
            action_pred, _ = policy(state) 
        dist = distributions.Categorical(logits = action_pred)
        action = dist.sample().item()

        next_state, reward, done, truncated, _ = env.step(action)
        buffer.store(state, action, next_state, reward, float(done), float(truncated))
        actor_loss, critic_loss = update_policy(policy, target_policy, optimizer, buffer, batch_size, discounted_factor, beta, weight_clip)
        
        soft_update(policy, target_policy, tau)

        state = next_state 
        episode_rewards += reward 
        episode_actor_loss += actor_loss 
        episode_critic_loss += critic_loss 

    return episode_actor_loss, episode_critic_loss, episode_rewards
    


In [11]:
def update_policy(policy, target_policy, optimizer, buffer,batch_size, discounted_factor, beta, weight_clip):
    policy.train()

    if buffer.size < batch_size:
        return 0.0, 0.0
    
    #1. Rút ngẫu nhiên sample từ buffer 
    states, actions, next_states, rewards, dones, truncateds = buffer.sample_batch(batch_size)
    
    actions_pred, values_pred = policy(states)
    values_pred = values_pred.squeeze(-1)

    with torch.no_grad():
        _, next_value_pred = target_policy(next_states)
        next_value_pred = next_value_pred.squeeze(-1)
        #TD error, giống như trong paper AWR, tác gỉa sử dụng TD thay vì GAE và monte 
        # TD = reward * (1 - done) + discounted_factor * V', khi terminate thif reward = 0 
        returns = rewards + discounted_factor * next_value_pred * (1 - dones) 
        #(1 - dones) CHỈ ĐƯỢC PHÉP dùng để nhân vào Tương lai (next_value_pred), với ý nghĩa là "game kết thúc rồi thì không còn giá trị tương lai nữa".

        # advantage 
        advantages = returns - values_pred
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        weights = torch.exp(1/beta * advantages) # nhu trong cong thuc. 
        weights = torch.clamp(weights, max=weight_clip) # chặn trên để tránh cập nhật quá nhiều, giống ratio trong ppo


    #chuan hoa returns 
    #returns = (returns - returns.mean()) / (returns.std() + 1e-8) đã qua test, nó làm bellman nổ

    # Đưa vào distribution
    dist = distributions.Categorical(logits = actions_pred)
    log_probs = dist.log_prob(actions)

    # tinh loss
    actor_loss = -(weights * log_probs).mean()
    critic_loss = F.mse_loss(returns, values_pred)
    loss = actor_loss + 0.5 * critic_loss 

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return actor_loss.item(), critic_loss.item()


In [12]:
def evaluate(env, policy):
    policy.eval()
    state, _ = env.reset()
    done, truncated = False, False 
    ep_reward = 0
    while not done and not truncated:
        
        with torch.no_grad():
            state  = torch.tensor(state).unsqueeze(0)
            action_pred, _ = policy(state)
        action = F.softmax(action_pred, dim = -1)
        action = torch.argmax(action_pred, dim = -1)
        state, reward, done, truncated, _ = env.step(action.item())
        ep_reward += reward 

    return ep_reward 

In [13]:
MAX_EPISODES = 500
DISCOUNT_FACTOR = 0.99
N_TRIALS = 25
REWARD_THRESHOLD = 500
PRINT_EVERY = 10

BUFFER_CAPACITY = 50000
BATCH_SIZE = 256
TAU = 0. 5
BETA = 0.99
WEIGHT_CLIP = 20.0

train_rewards = []
test_rewards = []


INPUT_DIM = train_env.observation_space.shape
HIDDEN_DIM = 128 
OUTPUT_DIM = train_env.action_space.n 


target_actor = MLP(INPUT_DIM[0], HIDDEN_DIM, OUTPUT_DIM)
target_critic = MLP(INPUT_DIM[0], HIDDEN_DIM, 1)
target_policy = ActorCritic(target_actor, target_critic)
target_policy.load_state_dict(policy.state_dict())
target_policy.eval()

buffer = ReplayBuffer(BUFFER_CAPACITY, INPUT_DIM)
for episode in range(1, MAX_EPISODES+1):
    
    policy_loss, value_loss, train_reward = train(train_env, policy, target_policy, optimizer, buffer, BATCH_SIZE, DISCOUNT_FACTOR, BETA, WEIGHT_CLIP, TAU)
    
    test_reward = evaluate(test_env, policy)
    
    train_rewards.append(train_reward)
    test_rewards.append(test_reward)
    
    mean_train_rewards = np.mean(train_rewards[-N_TRIALS:])
    mean_test_rewards = np.mean(test_rewards[-N_TRIALS:])
    
    if episode % PRINT_EVERY == 0:
    
        print(f'| Episode: {episode:3} | Mean Train Rewards: {mean_train_rewards:5.1f} | Mean Test Rewards: {mean_test_rewards:5.1f} |')
    
    if mean_test_rewards >= REWARD_THRESHOLD:
        
        print(f'Reached reward threshold in {episode} episodes')
        
        break

| Episode:  10 | Mean Train Rewards:  20.0 | Mean Test Rewards:   9.4 |
| Episode:  20 | Mean Train Rewards:  20.2 | Mean Test Rewards:   9.3 |
| Episode:  30 | Mean Train Rewards:  17.3 | Mean Test Rewards:   9.4 |
| Episode:  40 | Mean Train Rewards:  16.0 | Mean Test Rewards:  20.9 |
| Episode:  50 | Mean Train Rewards:  37.4 | Mean Test Rewards:  56.3 |
| Episode:  60 | Mean Train Rewards:  94.4 | Mean Test Rewards: 211.3 |
| Episode:  70 | Mean Train Rewards: 153.5 | Mean Test Rewards: 322.8 |
| Episode:  80 | Mean Train Rewards: 191.5 | Mean Test Rewards: 345.6 |
| Episode:  90 | Mean Train Rewards: 187.4 | Mean Test Rewards: 284.5 |
| Episode: 100 | Mean Train Rewards: 186.1 | Mean Test Rewards: 251.7 |
| Episode: 110 | Mean Train Rewards: 190.5 | Mean Test Rewards: 241.8 |
| Episode: 120 | Mean Train Rewards: 195.8 | Mean Test Rewards: 235.2 |
| Episode: 130 | Mean Train Rewards: 204.0 | Mean Test Rewards: 230.8 |
| Episode: 140 | Mean Train Rewards: 210.1 | Mean Test Rewards: 